In [1]:
# LLM Feature Generation for EV Charging User Personas
# This notebook creates secondary personas for users based on their charging patterns,
# geographic context, and activity using a local Ollama model

import pandas as pd
import numpy as np
import requests
import json
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Starting LLM Feature Generation for User Personas...")
print("=" * 60)


Starting LLM Feature Generation for User Personas...


# Step 1: Load the data
- charging data
- landmark data
- user persona data

In [ ]:
print("Loading datasets...")

# Load user personas
user_personas = pd.read_csv('./uploads/user_full_personas_generated.csv')
print(f"Loaded user personas: {len(user_personas)} records")

# Load charger persona correlation
charger_personas = pd.read_csv('./uploads/charger_persona_correlation_generated.csv')
print(f"Loaded charger personas: {len(charger_personas)} records")

# Load transaction data for frequency analysis
transaction_data = pd.read_csv('./uploads/transaction_data_tagged.csv')
print(f"Loaded transaction data: {len(transaction_data)} records")

# Load charger landmarks for geographic context
charger_landmarks = pd.read_csv('./uploads/charger_landmarks_heuristic.csv')
print(f"Loaded charger landmarks: {len(charger_landmarks)} records")

print("\nDataset overview:")
print("User Personas columns:", list(user_personas.columns))
print("Charger Personas columns:", list(charger_personas.columns))
print("Transaction Data columns:", list(transaction_data.columns))
print("Charger Landmarks columns:", list(charger_landmarks.columns))


Loading datasets...
Loaded user personas: 19944 records
Loaded charger personas: 1316 records
Loaded transaction data: 196582 records
Loaded charger landmarks: 6920 records

Dataset overview:
User Personas columns: ['CustomerId', 'primary_persona', 'secondary_persona']
Charger Personas columns: ['ChargeBoxId', 'primary_mode', 'secondary_mode', 'user_count']
Transaction Data columns: ['Created', 'ChargeId', 'ChargeBoxId', 'Current', 'Max Power', 'EvseId', 'SiteName', 'StartTime', 'StopTime', 'Duration', 'Energy', 'CustomerId', 'ProductName', 'SiteOwner', 'Contract', 'Zone', 'session_duration_hours', 'day_of_week', 'hour_of_day', 'is_weekend', 'user_loyalty_ratio', 'time_since_last_charge_hrs', 'persona']
Charger Landmarks columns: ['ChargeBoxId', 'LandmarkName', 'Category', 'Lat', 'Lon', 'Distance_m']


# Step 2: Analyze charger usage frequency
- get the **90th percentile** of chargers used by a user (most frequent)

In [ ]:
print("Analyzing charger usage patterns...")

def get_user_frequent_chargers(user_id: str, percentile: int = 90) -> list[int]:
    """
    Get the most frequent chargers for a user at the specified percentile
    Args:
        user_id: The ID of the user to get the frequent chargers for
        percentile: The percentile of the chargers to get (default is 90)
    Returns:
        A list of the most frequent chargers for the user
    """
    user_transactions = transaction_data[transaction_data['CustomerId'] == user_id]
    
    if len(user_transactions) == 0:
        return []
    
    # Count charger usage frequency
    charger_counts = user_transactions['ChargeBoxId'].value_counts()
    
    # Calculate the percentile threshold; get >= 90% of the chargers
    threshold = np.percentile(charger_counts.values, percentile)
    frequent_chargers = charger_counts[charger_counts >= threshold].index.tolist()
    
    return frequent_chargers

print("Computing frequent chargers for each user (this may take a moment)...")
user_frequent_chargers = {}

# Get unique users from transaction data
unique_users = transaction_data['CustomerId'].unique()
print(f"Processing {len(unique_users)} unique users...")

for i, user_id in enumerate(unique_users):
    if i % 1000 == 0:
        print(f"Processed {i}/{len(unique_users)} users...")
    
    frequent_chargers = get_user_frequent_chargers(user_id, percentile=90)
    user_frequent_chargers[user_id] = frequent_chargers

print(f"Completed analysis for {len(user_frequent_chargers)} users")

# Display some statistics
total_frequent_chargers = sum(len(chargers) for chargers in user_frequent_chargers.values())
avg_frequent_chargers = total_frequent_chargers / len(user_frequent_chargers) if user_frequent_chargers else 0
print(f"Average number of frequent chargers per user: {avg_frequent_chargers:.2f}")


Analyzing charger usage patterns...
Computing frequent chargers for each user (this may take a moment)...
Processing 19945 unique users...
Processed 0/19945 users...
Processed 1000/19945 users...
Processed 2000/19945 users...
Processed 3000/19945 users...
Processed 4000/19945 users...
Processed 5000/19945 users...
Processed 6000/19945 users...
Processed 7000/19945 users...
Processed 8000/19945 users...
Processed 9000/19945 users...
Processed 10000/19945 users...
Processed 11000/19945 users...
Processed 12000/19945 users...
Processed 13000/19945 users...
Processed 14000/19945 users...
Processed 15000/19945 users...
Processed 16000/19945 users...
Processed 17000/19945 users...
Processed 18000/19945 users...
Processed 19000/19945 users...
Completed analysis for 19945 users
Average number of frequent chargers per user: 1.30


# Step 3 : Attribute charger charactersistic
- **frequent usage pattern** analysis for each user

In [4]:
print("Attributing charger characteristics to users...")

def get_charger_attributes(charger_id: str) -> dict:
    """
    Get attributes for a specific charger
    Args:
        charger_id: The ID of the charger to get the attributes for
    Returns:
        A dictionary of the attributes for the charger
    """
    # Get charger persona information
    charger_info = charger_personas[charger_personas['ChargeBoxId'] == charger_id]
    
    # Get landmark information
    landmark_info = charger_landmarks[charger_landmarks['ChargeBoxId'] == charger_id]
    
    attributes = {}
    
    if not charger_info.empty:
        attributes['primary_mode'] = charger_info.iloc[0]['primary_mode']
        attributes['secondary_mode'] = charger_info.iloc[0]['secondary_mode']
        attributes['user_count'] = charger_info.iloc[0]['user_count']
    
    if not landmark_info.empty:
        # Get the most common category for this charger
        categories = landmark_info['Category'].value_counts()
        if not categories.empty:
            attributes['primary_category'] = categories.index[0]
            attributes['category_diversity'] = len(categories)
        
        # Get landmark names
        landmarks = landmark_info['LandmarkName'].unique()
        attributes['landmark_count'] = len(landmarks)
        attributes['primary_landmark'] = landmarks[0] if len(landmarks) > 0 else None
    
    return attributes

# Create user profiles with aggregated charger attributes
print("Building user profiles with charger attributes...")
user_profiles = []

for user_id, frequent_chargers in user_frequent_chargers.items():
    if not frequent_chargers:
        continue
    
    profile = {
        'CustomerId': user_id,
        'frequent_charger_count': len(frequent_chargers),
        'charger_ids': frequent_chargers
    }
    
    # Aggregate attributes from frequent chargers
    all_attributes = []
    for charger_id in frequent_chargers:
        attrs = get_charger_attributes(charger_id)
        if attrs:
            all_attributes.append(attrs)
    
    if all_attributes:
        # Aggregate primary modes
        primary_modes = [attr.get('primary_mode') for attr in all_attributes if attr.get('primary_mode')]
        if primary_modes:
            mode_counts = Counter(primary_modes)
            profile['dominant_primary_mode'] = mode_counts.most_common(1)[0][0]
            profile['primary_mode_diversity'] = len(mode_counts)
        
        # Aggregate secondary modes
        secondary_modes = [attr.get('secondary_mode') for attr in all_attributes if attr.get('secondary_mode')]
        if secondary_modes:
            sec_mode_counts = Counter(secondary_modes)
            profile['dominant_secondary_mode'] = sec_mode_counts.most_common(1)[0][0]
            profile['secondary_mode_diversity'] = len(sec_mode_counts)
        
        # Aggregate categories
        categories = [attr.get('primary_category') for attr in all_attributes if attr.get('primary_category')]
        if categories:
            cat_counts = Counter(categories)
            profile['dominant_category'] = cat_counts.most_common(1)[0][0]
            profile['category_diversity'] = len(cat_counts)
        
        # Calculate average user count (popularity of chargers)
        user_counts = [attr.get('user_count', 0) for attr in all_attributes if attr.get('user_count')]
        if user_counts:
            profile['avg_charger_popularity'] = np.mean(user_counts)
        
        # Count total landmarks
        landmark_counts = [attr.get('landmark_count', 0) for attr in all_attributes if attr.get('landmark_count')]
        if landmark_counts:
            profile['total_landmarks'] = sum(landmark_counts)
    
    user_profiles.append(profile)

# Convert to DataFrame
user_attributes_df = pd.DataFrame(user_profiles)
print(f"Created user attribute profiles for {len(user_attributes_df)} users")

# Display sample of user profiles
if not user_attributes_df.empty:
    print("\nSample user profiles:")
    print(user_attributes_df.head())
    
    print(f"\nAttribute summary:")
    print(f"Users with dominant primary mode: {user_attributes_df['dominant_primary_mode'].notna().sum()}")
    print(f"Users with dominant secondary mode: {user_attributes_df['dominant_secondary_mode'].notna().sum()}")
    print(f"Users with dominant category: {user_attributes_df['dominant_category'].notna().sum()}")


Attributing charger characteristics to users...
Building user profiles with charger attributes...
Created user attribute profiles for 19944 users

Sample user profiles:
       CustomerId  frequent_charger_count           charger_ids  \
0      00HLAS76E7                       1            [scdc0003]   
1  0434C50AD86380                       1            [scac0154]   
2  04493232C05B85                       2  [scdc0006, scdc0012]   
3  045FFA0AD86380                       1            [scdc0021]   
4  0490C70AD86380                       2  [scdc0023, scdc0006]   

  dominant_primary_mode  primary_mode_diversity dominant_secondary_mode  \
0                 noise                       1                   other   
1                 noise                       1             residential   
2               regular                       1                   other   
3               regular                       1                   other   
4               regular                       1      

# Step 4 : Analyze using sandboxed LLMs
- **sandboxing required** to avoid data going out of the system
- use a capable model with **world knowledge** that can be run locally safely
- use **ollama** to pull a competent small LLM (e.g. gemma3-12b | qwen3-14b | deepseek-r1-8b) etc.

In [11]:
import requests
import logging

#Use a safe, sandboxed LLM model running locally with Ollama to generate secondary personas

class OllamaClient:
    def __init__(self,
                #  model_name="deepseek-r1:0528-8b",
				#  model_name="gemma3:12b",
				 model_name="gemma3:4b",
                 base_url="http://localhost:11434"):
        
        self.model_name = model_name
        self.base_url = base_url
        self.api_url = f"{base_url}/api/generate"

        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        self.logger.addHandler(logging.StreamHandler())

        self.logger.info(f"Initialized OllamaClient with model: {self.model_name}")
    
    def test_connection(self):
        """
        Test if Ollama is running and model is available
        Args:
            model_name: The name of the model to test
            base_url: The base URL of the Ollama server
        Returns:
            True if the connection is successful, False otherwise
        """
        try:
            response = requests.post(self.api_url, json={
                "model": self.model_name,
                "prompt": "Hello, are you working?",
                "stream": False
            }, timeout=120)
            
            if response.status_code == 200:
                self.logger.info(f"Successfully connected to Ollama with model: {self.model_name}")
                return True
            else:
                self.logger.error(f"Connection failed. Status code: {response.status_code}")
                print(f"Response: {response.text}")
                return False
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Connection error: {e}")
            self.logger.error("Make sure Ollama is running with: ollama serve")
            self.logger.error(f"And the model is available with: ollama pull {self.model_name}")
            return False
        
    def generate(self,
                 prompt: str,
                 **kwargs: dict):
        """
        Generate a response from the LLM
        Args:
            prompt: The prompt to generate a response for
            options: A dictionary containing the options for the LLM
        Returns:
            A string containing the generated response
        """
        try:
            response = requests.post(self.api_url, json={
                "model": self.model_name,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": kwargs.get("temperature", 0.7),
                    "max_tokens": kwargs.get("max_tokens", 20)
                }
            }, timeout=60)

            if response.status_code == 200:
                result = response.json()
                return result.get('response', '').strip()
            else:
                self.logger.error(f"Error generating response: {response.status_code}")
                return "indeterminate"
        except Exception as e:
            self.logger.error(f"Error in response generation: {e}")
            return "indeterminate"
        
    def batch_generate(self,
                       prompts: list[str],
                       **kwargs: dict) -> list[str]:
        """
        Generate a batch of responses from the LLM
        Args:
            prompts: A list of prompts to generate responses for
            kwargs: A dictionary containing the options for the LLM
        Returns:
            A list of strings containing the generated responses
        """
        responses = []
        for i, prompt in enumerate(prompts):
            
            try:
                response = self.generate(prompt, **kwargs)
                
            except Exception as e:
                self.logger.warning(f"Error in batch response generation: {e}")
                self.logger.warning(f"Prompt: {prompt} | Index: {i}\n\n")
                response = "indeterminate"
            
            finally:
                responses.append(response)
        
        self.logger.info(f"Batch response generation completed with {len(responses)} responses")
        return responses
    
    def generate_persona(self, user_context):
        """
        Generate a secondary persona using the LLM
        Args:
            user_context: A dictionary containing the user context
        Returns:
            A string containing the generated persona name
        """
        prompt = f"""
Based on the following EV charging user profile, create a unique and descriptive secondary persona name that captures their charging behavior, location preferences, and activity patterns.
NOTE: You are looking at a user profile, not a charger profile.

User Profile:
- Primary persona: {user_context.get('primary_persona', 'Unknown')}
- Dominant charging location category: {user_context.get('dominant_category', 'Unknown')}
- Charging mode preference: {user_context.get('dominant_secondary_mode', 'Unknown')}
- Location diversity: {user_context.get('category_diversity', 0)} different types of locations
- Charger popularity preference: {user_context.get('avg_charger_popularity', 0):.1f} average users per charger
- Charging frequency: {user_context.get('frequent_charger_count', 0)} frequent chargers
- Geographic landmarks: {user_context.get('total_landmarks', 0)} total landmarks

Consider these factors:
1. Geographic context (residential, commercial, shopping, etc.)
2. Charging behavior patterns (regular vs occasional, popular vs niche locations)
3. Location diversity (single-location vs multi-location user)
4. Activity context based on landmark types

Create a concise, descriptive persona name (2-4 words) that captures the essence of this user's charging lifestyle. Examples might include: "Urban Commuter", "Weekend Explorer", "Mall Shopper", "Home-Base Charger", "Multi-Zone Traveler", etc.

Respond with only the persona name, nothing else.
"""
        
        try:
            response = requests.post(self.api_url, json={
                "model": self.model_name,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.7,
                    "max_tokens": 20
                }
            }, timeout=60)
            
            if response.status_code == 200:
                result = response.json()
                return result.get('response', '').strip()
            else:
                print(f"Error generating persona: {response.status_code}")
                return "Unknown Persona"
        except Exception as e:
            print(f"Error in persona generation: {e}")
            return "Unknown Persona"

# Initialize Ollama client
ollama_client = OllamaClient()

# Test the connection
print("Testing Ollama connection...")
connection_success = ollama_client.test_connection()

if not connection_success:
    print("\nWARNING: Ollama connection failed!")
    print("To fix this:")
    print("1. Make sure Ollama is installed and running: ollama serve")
    print("2. Pull the required model: ollama pull gemma3:12b")
    print("3. Check if the model name is correct")
    print("\nYou can continue with mock data for testing purposes.")

Initialized OllamaClient with model: gemma3:4b
Initialized OllamaClient with model: gemma3:4b


Initialized OllamaClient with model: gemma3:4b


Testing Ollama connection...


Successfully connected to Ollama with model: gemma3:4b
Successfully connected to Ollama with model: gemma3:4b
Successfully connected to Ollama with model: gemma3:4b


# Step 5 - Generate Secondary Personas using the LLM
- detailed personas per user-ID give us insight into type of customers for DR pricing
- further usage can be derived with specific user statistical reports
- high potential for sandboxed experimental guesses for treating each customer differently during DR events

In [7]:
print("Generating secondary personas using LLM...")

def get_user_activity_context(user_id: str) -> dict:
    """
    Extract activity context from transaction data
    Args:
        user_id: The ID of the user to get the activity context for
    Returns:
        A dictionary containing the activity context
    """
    user_transactions = transaction_data[transaction_data['CustomerId'] == user_id]
    
    if user_transactions.empty:
        return {}
    
    context = {
        'total_sessions': len(user_transactions),
        'avg_session_duration': user_transactions['session_duration_hours'].mean(),
        'avg_energy_consumption': user_transactions['Energy'].mean(),
        'weekend_ratio': user_transactions['is_weekend'].mean(),
        'peak_hour': user_transactions['hour_of_day'].mode().iloc[0] if not user_transactions['hour_of_day'].mode().empty else 12,
        'loyalty_score': user_transactions['user_loyalty_ratio'].mean(),
        'zone_diversity': user_transactions['Zone'].nunique(),
        'primary_zone': user_transactions['Zone'].mode().iloc[0] if not user_transactions['Zone'].mode().empty else 'Unknown'
    }
    
    return context

# Merge user personas with user attributes and activity context
print("Preparing comprehensive user profiles...")
enhanced_user_profiles = []

# Create a lookup dictionary for user attributes
user_attr_dict = user_attributes_df.set_index('CustomerId').to_dict('index')

for _, user in user_personas.iterrows():
    user_id = user['CustomerId']
    
    # Get existing persona info
    profile = {
        'CustomerId': user_id,
        'primary_persona': user['primary_persona'],
        'existing_secondary_persona': user['secondary_persona']
    }
    
    # Add charger attributes if available
    if user_id in user_attr_dict:
        profile.update(user_attr_dict[user_id])
    
    # Add activity context
    activity_context = get_user_activity_context(user_id)
    profile.update(activity_context)
    
    enhanced_user_profiles.append(profile)

enhanced_profiles_df = pd.DataFrame(enhanced_user_profiles)
print(f"Created enhanced profiles for {len(enhanced_profiles_df)} users")

# Display sample enhanced profile
if not enhanced_profiles_df.empty:
    print("\nSample enhanced user profile:")
    sample_user = enhanced_profiles_df.iloc[0].to_dict()
    for key, value in sample_user.items():
        print(f"  {key}: {value}")

print(f"\nEnhanced profile columns: {list(enhanced_profiles_df.columns)}")

Generating secondary personas using LLM...
Preparing comprehensive user profiles...
Created enhanced profiles for 19944 users

Sample enhanced user profile:
  CustomerId: 00HLAS76E7
  primary_persona: regular
  existing_secondary_persona: other
  frequent_charger_count: 1
  charger_ids: ['scdc0003']
  dominant_primary_mode: noise
  primary_mode_diversity: 1
  dominant_secondary_mode: other
  secondary_mode_diversity: 1
  dominant_category: other
  category_diversity: 1.0
  avg_charger_popularity: 335.0
  total_landmarks: 1.0
  total_sessions: 10
  avg_session_duration: 0.6285555555555555
  avg_energy_consumption: 22.6171
  weekend_ratio: 0.2
  peak_hour: 13.0
  loyalty_score: 0.8
  zone_diversity: 2
  primary_zone: Central

Enhanced profile columns: ['CustomerId', 'primary_persona', 'existing_secondary_persona', 'frequent_charger_count', 'charger_ids', 'dominant_primary_mode', 'primary_mode_diversity', 'dominant_secondary_mode', 'secondary_mode_diversity', 'dominant_category', 'categor

# Step 6 : Generating personas
- **optimize** first for personas where we have enough data
- **compute optimization** for running data through the LLM

In [12]:
print("Generating LLM-based secondary personas...")

# Filter users with sufficient data for persona generation
users_for_llm = enhanced_profiles_df[
    enhanced_profiles_df['frequent_charger_count'].notna() & 
    (enhanced_profiles_df['frequent_charger_count'] > 0) &
    enhanced_profiles_df['total_sessions'].notna() &
    (enhanced_profiles_df['total_sessions'] > 0)
].copy()

print(f"Users eligible for LLM persona generation: {len(users_for_llm)}")


Generating LLM-based secondary personas...
Users eligible for LLM persona generation: 19944


In [19]:
import time

def create_fallback_persona(user_profile):
    """Create a rule-based persona when LLM is not available"""
    category = user_profile.get('dominant_category', 'unknown')
    mode = user_profile.get('dominant_secondary_mode', 'unknown')
    zone = user_profile.get('primary_zone', 'unknown')
    diversity = user_profile.get('category_diversity', 0)
    
    # Rule-based persona generation
    if category == 'residential':
        if diversity > 2:
            return "Multi-Zone Resident"
        else:
            return "Home-Base Charger"
    elif category == 'commercial':
        return "Business District User"
    elif 'shopping' in str(mode).lower() or 'mall' in str(mode).lower():
        return "Mall Shopper"
    elif zone == 'Central':
        return "Urban Commuter"
    elif diversity > 3:
        return "Multi-Location Traveler"
    else:
        return "Regular Charger"

# Generate personas
print("Starting persona generation...")
generated_personas = []

# Process users in batches to avoid overwhelming the LLM
batch_size = 20
total_batches = len(users_for_llm) // batch_size + (1 if len(users_for_llm) % batch_size > 0 else 0)

for batch_idx in range(total_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(users_for_llm))
    batch_users = users_for_llm.iloc[start_idx:end_idx]
    
    print(f"Processing batch {batch_idx + 1}/{total_batches} ({len(batch_users)} users)...")
    
    for _, user_profile in batch_users.iterrows():
        user_context = user_profile.to_dict()
        
        if connection_success:
            # Use LLM to generate persona
            try:
                llm_persona = ollama_client.generate_persona(user_context)
                # Clean up the response
                llm_persona = llm_persona.strip().strip('"').strip("'")
                if llm_persona and llm_persona != "Unknown Persona":
                    generated_persona = llm_persona
                else:
                    generated_persona = create_fallback_persona(user_context)
            except Exception as e:
                print(f"Error generating persona for user {user_profile['CustomerId']}: {e}")
                generated_persona = create_fallback_persona(user_context)
        else:
            # Use fallback rule-based approach
            generated_persona = create_fallback_persona(user_context)
        
        generated_personas.append({
            'CustomerId': user_profile['CustomerId'],
            'llm_secondary_persona': generated_persona
        })
    
    # Small delay between batches to avoid overwhelming the LLM
    if connection_success and batch_idx < total_batches - 1:
        time.sleep(1)

	#save batched results after every 100 batches
    if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == total_batches:
        temp_df = pd.DataFrame(generated_personas)
        temp_df.to_csv(f'./Outputs/llm_personas_batch_{batch_idx + 1}.csv', index=False)
        print(f"Saved batch results to llm_personas_batch_{batch_idx + 1}.csv")

        del temp_df

    # print("Results from current batch:")
    # for persona in generated_personas[-5:]:
    #     print(f"  UserID: {persona['CustomerId']} | Persona: {persona['llm_secondary_persona']}")

    # break

# Convert to DataFrame
llm_personas_df = pd.DataFrame(generated_personas)
print(f"Generated {len(llm_personas_df)} LLM-based personas")

# Display persona distribution
if not llm_personas_df.empty:
    print(f"\nGenerated persona distribution:")
    persona_counts = llm_personas_df['llm_secondary_persona'].value_counts()
    print(persona_counts.head(10))

Starting persona generation...
Processing batch 1/998 (20 users)...
Processing batch 2/998 (20 users)...
Processing batch 3/998 (20 users)...
Processing batch 4/998 (20 users)...
Processing batch 5/998 (20 users)...
Processing batch 6/998 (20 users)...
Processing batch 7/998 (20 users)...
Processing batch 8/998 (20 users)...
Processing batch 9/998 (20 users)...
Processing batch 10/998 (20 users)...
Processing batch 11/998 (20 users)...
Processing batch 12/998 (20 users)...
Processing batch 13/998 (20 users)...
Processing batch 14/998 (20 users)...
Processing batch 15/998 (20 users)...
Processing batch 16/998 (20 users)...
Processing batch 17/998 (20 users)...
Processing batch 18/998 (20 users)...
Processing batch 19/998 (20 users)...
Processing batch 20/998 (20 users)...
Processing batch 21/998 (20 users)...
Processing batch 22/998 (20 users)...
Processing batch 23/998 (20 users)...
Processing batch 24/998 (20 users)...
Processing batch 25/998 (20 users)...
Processing batch 26/998 (20 

# Step 7: Collate personas, save results

In [ ]:
print("Creating final dataset with LLM-generated secondary personas...")

# Merge original user personas with LLM-generated personas
final_user_personas = user_personas.merge(
    llm_personas_df, 
    on='CustomerId', 
    how='left'
)

# Fill missing LLM personas with fallback for users not processed
users_without_llm_persona = final_user_personas['llm_secondary_persona'].isna()
print(f"Users without LLM persona: {users_without_llm_persona.sum()}")

if users_without_llm_persona.sum() > 0:
    print("Generating fallback personas for remaining users...")
    
    for idx, row in final_user_personas[users_without_llm_persona].iterrows():
        user_id = row['CustomerId']
        
        # Get user context for fallback
        user_context = {}
        if user_id in user_attr_dict:
            user_context.update(user_attr_dict[user_id])
        
        activity_context = get_user_activity_context(user_id)
        user_context.update(activity_context)
        
        fallback_persona = create_fallback_persona(user_context)
        final_user_personas.loc[idx, 'llm_secondary_persona'] = fallback_persona

# Create the final secondary persona column
final_user_personas['secondary_persona_llm'] = final_user_personas['llm_secondary_persona']

print(f"Final dataset shape: {final_user_personas.shape}")
print(f"Columns: {list(final_user_personas.columns)}")

# Display final persona statistics
print("\nFinal LLM Secondary Persona Distribution:")
persona_distribution = final_user_personas['secondary_persona_llm'].value_counts()
print(persona_distribution.head(15))

print(f"\nTotal unique LLM secondary personas: {final_user_personas['secondary_persona_llm'].nunique()}")
print(f"Coverage: {(final_user_personas['secondary_persona_llm'].notna().sum() / len(final_user_personas) * 100):.1f}%")

# Compare with original secondary personas
print("\nComparison with original secondary personas:")
print("Original secondary personas:", final_user_personas['secondary_persona'].value_counts().head(10))
print("\nLLM secondary personas:", final_user_personas['secondary_persona_llm'].value_counts().head(10))


Creating final dataset with LLM-generated secondary personas...
Users without LLM persona: 0
Final dataset shape: (19944, 5)
Columns: ['CustomerId', 'primary_persona', 'secondary_persona', 'llm_secondary_persona', 'secondary_persona_llm']

Final LLM Secondary Persona Distribution:
secondary_persona_llm
Neighborhood Home Charger      4695
Suburban Home Charger          2953
Mall Route Charger             2283
Home-Neighborhood Charger       997
Tech Park Navigator             854
Suburban Home Hub               820
Urban Hub Charger               591
Regional Hub Charger            585
Home-Base Suburban Charger      477
Mall Momentum Driver            464
Mall Route Navigator            450
Urban Explorer Charger          373
Home-Neighborhood Navigator     352
Mall Navigator                  346
Mall Frequent Charger           292
Name: count, dtype: int64

Total unique LLM secondary personas: 158
Coverage: 100.0%

Comparison with original secondary personas:
Original secondary person

# Step 8: Save secondary personas

In [21]:
print("Saving results...")

# Save the final dataset with LLM personas
output_file = './Outputs/user_personas_llm_enhanced.csv'
final_user_personas.to_csv(output_file, index=False)
print(f"✅ Saved enhanced user personas to: {output_file}")

# Save detailed user profiles for further analysis
detailed_profiles_file = './Outputs/user_detailed_profiles.csv'
enhanced_profiles_df.to_csv(detailed_profiles_file, index=False)
print(f"✅ Saved detailed user profiles to: {detailed_profiles_file}")

# Create a summary report
summary_stats = {
    'total_users_processed': len(final_user_personas),
    'users_with_llm_personas': final_user_personas['secondary_persona_llm'].notna().sum(),
    'unique_llm_personas': final_user_personas['secondary_persona_llm'].nunique(),
    'users_with_frequent_chargers': len(users_for_llm),
    'avg_frequent_chargers_per_user': user_attributes_df['frequent_charger_count'].mean() if not user_attributes_df.empty else 0,
    'ollama_connection_successful': connection_success,
    'model_used': ollama_client.model_name
}

print("\n" + "="*60)
print("LLM FEATURE GENERATION SUMMARY")
print("="*60)
for key, value in summary_stats.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

print("\nTop 10 LLM-Generated Secondary Personas:")
print("-" * 40)
top_personas = final_user_personas['secondary_persona_llm'].value_counts().head(10)
for persona, count in top_personas.items():
    percentage = (count / len(final_user_personas)) * 100
    print(f"{persona}: {count} users ({percentage:.1f}%)")

# Sample user profiles for verification
print(f"\nSample User Profiles:")
print("-" * 40)
sample_users = final_user_personas.head(3)
for _, user in sample_users.iterrows():
    print(f"User ID: {user['CustomerId']}")
    print(f"  Primary Persona: {user['primary_persona']}")
    print(f"  Original Secondary: {user['secondary_persona']}")
    print(f"  LLM Secondary: {user['secondary_persona_llm']}")
    print()

print("✅ LLM Feature Generation completed successfully!")
print("\nNext steps:")
print("1. Review the generated personas in the output files")
print("2. Validate the persona assignments with domain experts")
print("3. Use the enhanced personas for downstream analysis")
print("4. Consider fine-tuning the LLM prompts based on results")

Saving results...
✅ Saved enhanced user personas to: ./Outputs/user_personas_llm_enhanced.csv
✅ Saved detailed user profiles to: ./Outputs/user_detailed_profiles.csv

LLM FEATURE GENERATION SUMMARY
Total Users Processed: 19944
Users With Llm Personas: 19944
Unique Llm Personas: 158
Users With Frequent Chargers: 19944
Avg Frequent Chargers Per User: 1.2986361813076615
Ollama Connection Successful: True
Model Used: gemma3:4b

Top 10 LLM-Generated Secondary Personas:
----------------------------------------
Neighborhood Home Charger: 4695 users (23.5%)
Suburban Home Charger: 2953 users (14.8%)
Mall Route Charger: 2283 users (11.4%)
Home-Neighborhood Charger: 997 users (5.0%)
Tech Park Navigator: 854 users (4.3%)
Suburban Home Hub: 820 users (4.1%)
Urban Hub Charger: 591 users (3.0%)
Regional Hub Charger: 585 users (2.9%)
Home-Base Suburban Charger: 477 users (2.4%)
Mall Momentum Driver: 464 users (2.3%)

Sample User Profiles:
----------------------------------------
User ID: 00HLAS76E7
  